# 07. Тесты на структурный сдвиг и различие коэффициентов

Воспроизводит численные результаты, использованные в работе:

- **§2.2.5** — тест Чоу и Quandt-Andrews для модели М3 на дате 2020-01-05.
- **§3.2.3** — t-тест на равенство коэффициентов между подпериодами 2017–2019 и 2020–2025.
- **§3.6 (B3, B7)** — поглощение коэффициентов при VIX и DXY через S&P 500 (М2 с/без $r_{SP500}$).

Все расчёты выполняются на `data/processed/merged_weekly.csv`. Стандартные ошибки в части t-теста — HAC (Newey-West).

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

df = (
    pd.read_csv('../data/processed/merged_weekly.csv', parse_dates=['date'])
      .sort_values('date')
      .reset_index(drop=True)
)

Y_VAR = 'r_btc'
CRYPTO_VARS = ['r_btc_lag', 'log_volume_btc', 'google_trends']
EXTERNAL_VARS = ['r_sp500', 'vix', 'r_dxy']
ALL_VARS = CRYPTO_VARS + EXTERNAL_VARS

df = df.dropna(subset=[Y_VAR] + ALL_VARS).reset_index(drop=True)
BREAK_DATE = pd.Timestamp('2020-01-05')
idx_break = (df['date'] >= BREAK_DATE).idxmax()

n = len(df)
n1, n2 = idx_break, n - idx_break
print(f'n = {n} | n1 (2017–2019) = {n1} | n2 (2020–2025) = {n2}')
print(f'Период: {df["date"].iloc[0].date()} – {df["date"].iloc[-1].date()}')
print(f'Точка разбивки: {BREAK_DATE.date()}')

n = 422 | n1 (2017–2019) = 153 | n2 (2020–2025) = 269
Период: 2017-01-29 – 2025-02-23
Точка разбивки: 2020-01-05


## 1. Тест Чоу для известной точки разбивки

Проверяет нулевую гипотезу о равенстве всех коэффициентов модели до и после `BREAK_DATE`. Применяется к спецификациям **М3** (полная) и **М2** (только внешние факторы).

$$F = \frac{(RSS_p - RSS_1 - RSS_2)/k}{(RSS_1 + RSS_2)/(n - 2k)} \sim F(k,\ n - 2k).$$

In [2]:
def rss(y, X):
    return float(np.sum(sm.OLS(y, X).fit().resid ** 2))

def chow_test(df, regressors, idx_break, label):
    y = df[Y_VAR].values
    X = sm.add_constant(df[regressors].values)
    n_, k = X.shape
    rss_p = rss(y, X)
    rss_1 = rss(y[:idx_break], X[:idx_break])
    rss_2 = rss(y[idx_break:], X[idx_break:])
    F = ((rss_p - rss_1 - rss_2) / k) / ((rss_1 + rss_2) / (n_ - 2 * k))
    p = 1 - stats.f.cdf(F, k, n_ - 2 * k)
    print(f'{label:25s} | F({k}, {n_ - 2*k}) = {F:.4f} | p = {p:.4f}')
    return F, p

_ = chow_test(df, ALL_VARS, idx_break, 'M3 (полная)')
_ = chow_test(df, EXTERNAL_VARS, idx_break, 'M2 (внешние факторы)')

M3 (полная)               | F(7, 408) = 0.7729 | p = 0.6103
M2 (внешние факторы)      | F(4, 414) = 0.8028 | p = 0.5239


## 2. Тест Quandt-Andrews

Перебор F-статистики Чоу по всем точкам в окне $[0{,}15T;\ 0{,}85T]$, фиксируется супремум. Применяется к двум вариантам:

- **(a) Все коэффициенты M3** меняются на дате $i$ (стандартный Quandt-Andrews).
- **(b) Только коэффициент при $r_{SP500}$** меняется (через interaction $D_i \cdot r_{SP500}$, $k = 1$ ограничение).

In [3]:
TRIM = 0.15
i_lo = int(np.ceil(TRIM * n))
i_hi = int(np.floor((1 - TRIM) * n))

y = df[Y_VAR].values
X = sm.add_constant(df[ALL_VARS].values)
k = X.shape[1]
rss_p = rss(y, X)

F_full, F_sp = [], []
for i in range(i_lo, i_hi + 1):
    rss_a = rss(y[:i], X[:i])
    rss_b = rss(y[i:], X[i:])
    F_full.append(((rss_p - rss_a - rss_b) / k) / ((rss_a + rss_b) / (n - 2 * k)))
    D = (np.arange(n) >= i).astype(float)
    Xi = np.column_stack([X, D * df['r_sp500'].values])
    rss_i = float(np.sum(sm.OLS(y, Xi).fit().resid ** 2))
    F_sp.append(((rss_p - rss_i) / 1) / (rss_i / (n - X.shape[1] - 1)))

for label, arr, k_break in [
    ('Sup-F полная M3 (k=7)', np.array(F_full), 7),
    ('Sup-F r_sp500 (k=1)', np.array(F_sp), 1),
]:
    j = int(np.nanargmax(arr))
    print(f'{label:25s} | Sup-F = {arr[j]:.4f} | дата = {df["date"].iloc[i_lo + j].date()}')

print('\nКритические значения Andrews (1993, trim = 0.15):')
print('  k = 7: 10% = 23.66 | 5% = 26.42 | 1% = 32.22')
print('  k = 1: 10% =  7.17 | 5% =  8.85 | 1% = 12.16')

Sup-F полная M3 (k=7)     | Sup-F = 2.8432 | дата = 2018-04-22
Sup-F r_sp500 (k=1)       | Sup-F = 3.6256 | дата = 2020-11-08

Критические значения Andrews (1993, trim = 0.15):
  k = 7: 10% = 23.66 | 5% = 26.42 | 1% = 32.22
  k = 1: 10% =  7.17 | 5% =  8.85 | 1% = 12.16


## 3. t-тест на равенство коэффициентов между подпериодами

Оценивает $t = (\hat\beta_2 - \hat\beta_1) / \sqrt{SE_1^2 + SE_2^2}$ для каждого регрессора М3 при HAC-стандартных ошибках. Степени свободы $n_1 + n_2 - 2k$. Соответствует таблице 3.5 в работе.

In [4]:
def hac_lag(T):
    return int(np.floor(4 * (T / 100) ** (2 / 9)))

mask1 = df['date'] < BREAK_DATE
mask2 = ~mask1

def fit_hac(mask, regressors):
    y_ = df.loc[mask, Y_VAR].values
    X_ = sm.add_constant(df.loc[mask, regressors].values)
    return sm.OLS(y_, X_).fit(cov_type='HAC', cov_kwds={'maxlags': hac_lag(len(y_))})

m1 = fit_hac(mask1, ALL_VARS)
m2 = fit_hac(mask2, ALL_VARS)

names = ['const'] + ALL_VARS
df_t = n1 + n2 - 2 * len(names)
rows = []
for i, name in enumerate(names):
    b1, b2 = m1.params[i], m2.params[i]
    se_d = np.sqrt(m1.bse[i] ** 2 + m2.bse[i] ** 2)
    t = (b2 - b1) / se_d
    p = 2 * (1 - stats.t.cdf(abs(t), df=df_t))
    rows.append({'param': name, 'beta_1': b1, 'beta_2': b2,
                 'delta': b2 - b1, 'SE_delta': se_d, 't': t, 'p': p})

result = pd.DataFrame(rows).set_index('param').round(4)
print(f'HAC-лаги: подпериод 1 — h = {hac_lag(n1)}, подпериод 2 — h = {hac_lag(n2)} | df = {df_t}')
result

HAC-лаги: подпериод 1 — h = 4, подпериод 2 — h = 4 | df = 408


,beta_1,beta_2,delta,SE_delta,t,p
param,,,,,,
const,0.1503,0.0005,-0.1498,0.1301,-1.1520,0.2500
r_btc_lag,-0.0257,0.0735,0.0992,0.0994,0.9978,0.3190
log_volume_btc,-0.0071,-0.0010,0.0061,0.0084,0.7265,0.4680
google_trends,0.0003,0.0003,0.0000,0.0010,0.0149,0.9881
r_sp500,0.3394,0.9514,0.6120,0.7700,0.7949,0.4271
vix,-0.0030,0.0002,0.0032,0.0027,1.1719,0.2419
r_dxy,-0.6106,0.4012,1.0118,1.5529,0.6515,0.5151


## 4. Поглощение VIX и DXY через S&P 500

Сравнение коэффициентов в трёх спецификациях М2:

- полная М2 ($r_{SP500} + VIX + r_{DXY}$);
- М2 без $r_{SP500}$ — какой коэффициент имели бы VIX и $r_{DXY}$ в одиночку;
- М2 без $r_{DXY}$ — устойчивость коэффициента при $r_{SP500}$ к исключению DXY (B7).

Стандартные ошибки HAC с лагом 5.

In [5]:
def fit_full(regressors, h=5):
    y_ = df[Y_VAR].values
    X_ = sm.add_constant(df[regressors].values)
    return sm.OLS(y_, X_).fit(cov_type='HAC', cov_kwds={'maxlags': h})

specs = {
    'M2 полная (SP500+VIX+DXY)': ['r_sp500', 'vix', 'r_dxy'],
    'M2 без SP500 (VIX+DXY)':    ['vix', 'r_dxy'],
    'M2 без DXY (SP500+VIX)':    ['r_sp500', 'vix'],
}

out = {}
for label, regs in specs.items():
    m = fit_full(regs)
    coefs = {name: (m.params[i + 1], m.pvalues[i + 1]) for i, name in enumerate(regs)}
    out[label] = coefs

table = pd.DataFrame({
    label: {f'{name}': f'{b:.5f} (p={p:.4f})'
            for name, (b, p) in coefs.items()}
    for label, coefs in out.items()
}).T
print('Коэффициент (HAC p-значение):')
table

Коэффициент (HAC p-значение):


,r_sp500,vix,r_dxy
M2 полная (SP500+VIX+DXY),0.79296 (p=0.0010),-0.00061 (p=0.3779),-0.09063 (p=0.8925)
M2 без SP500 (VIX+DXY),NaN,-0.00103 (p=0.2045),-0.96574 (p=0.1006)
M2 без DXY (SP500+VIX),0.80681 (p=0.0004),-0.00061 (p=0.3765),NaN


## Выводы

1. **Тест Чоу** на M3 и M2 не отвергает стабильность всех коэффициентов на 10% уровне (`F = 0.77 / 0.80`). Это согласуется с §2.2.5.
2. **Quandt-Andrews** Sup-F далеко от критических значений Andrews (1993) как для полной М3, так и для одиночного коэффициента $r_{SP500}$. Максимум для одиночного коэффициента приходится на ноябрь 2020 года, что качественно согласуется с гипотезой о растущей интеграции с фондовым рынком.
3. **t-тест на различие коэффициентов** (таблица 3.5): ни для одного регрессора нулевая гипотеза не отвергается на 10% уровне; для $r_{SP500}$ `t ≈ 0.79`, `p ≈ 0.43`. Это связано с большой стандартной ошибкой коэффициента в раннем подпериоде (`SE ≈ 0.74`).
4. **Поглощение VIX/DXY через S&P 500**: при включении $r_{SP500}$ оба коэффициента сжимаются по модулю и теряют значимость, что подтверждает интерпретацию VIX и DXY как мер риска, частично перекрывающихся с фондовым рынком. Исключение $r_{DXY}$ из М2 не меняет коэффициент при $r_{SP500}$ в третьем знаке.